# Bolo 04: cartoon voices (Indic Parler TTS)

Needs the `HF_TOKEN` secret. GPU T4. Output: `/kaggle/working/narration/<id>.wav` (best take) and `takes/`, plus `scores.json`.

In [ ]:
!pip install -q git+https://github.com/huggingface/parler-tts.git soundfile jiwer

In [ ]:
import os
os.environ["USE_TF"] = "0"; os.environ["USE_TORCH"] = "1"; os.environ["USE_JAX"] = "0"; os.environ["USE_FLAX"] = "0"

In [ ]:
import json, os
WORK = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'

# Indic Parler-TTS is gated: accept its terms on huggingface.co first, then add a READ token as a secret
# named HF_TOKEN (Kaggle: Add-ons -> Secrets; Colab: key icon -> Secrets).
token, errs = None, []
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    errs.append(f"kaggle: {type(e).__name__}: {e}")
if not token and not os.path.exists("/kaggle"):
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception as e:
        errs.append(f"colab: {type(e).__name__}: {e}")
token = token or os.environ.get("HF_TOKEN")
if token:
    from huggingface_hub import login
    login(token=token)
    MODEL = "ai4bharat/indic-parler-tts"
else:
    # API-started runs cannot read Kaggle secrets: run a pipeline SMOKE TEST with the ungated Parler-TTS mini
    # (same codebase/architecture) and stop. The real run must be started from the Kaggle UI.
    print("NO HF_TOKEN:", " | ".join(errs), "-> smoke-test mode with parler-tts/parler-tts-mini-v1")
    MODEL = "parler-tts/parler-tts-mini-v1"
SMOKE = MODEL != "ai4bharat/indic-parler-tts"

In [ ]:
SPEC = {
 "voices": {},
 "lines": [
  {
   "id": "n03",
   "voice": "narrator",
   "text": "Say hello to Aarav's new assistant. It is small enough to live on his laptop, and it understands Hinglish.",
   "say": [
    "Say hello to Aarav's new assistant. It is small enough to live on his laptop, and it understands Hinglish."
   ],
   "desc": "Mary speaks in a bright, excited, highly expressive voice, slightly high pitched and a bit fast, like revealing a surprise, with very clear audio and no background noise."
  },
  {
   "id": "n03b",
   "voice": "narrator",
   "text": "Not a fan of the name on the tag? Give it your own.",
   "say": [
    "Not a fan of the name on the tag? Give it your own."
   ],
   "desc": "Mary speaks in a playful, teasing, highly expressive voice, slightly high pitched and a bit fast, with very clear audio and no background noise."
  },
  {
   "id": "n04",
   "voice": "narrator",
   "text": "It writes down what he said, works out what he wants, and checks that the folder really exists. Then the terminal opens.",
   "say": [
    "It writes down what he said, works out what he wants, and checks that the folder really exists. Then the terminal opens."
   ],
   "desc": "Mary speaks in a lively, animated, highly expressive voice, slightly fast, with rising energy, with very clear audio and no background noise."
  },
  {
   "id": "n09",
   "voice": "narrator",
   "text": "Built on Bolo. Open source, and free. Try it, and tell us where it fails.",
   "say": [
    "Built on Bolo. Open source, and free. Try it, and tell us where it fails."
   ],
   "desc": "Mary speaks in an upbeat, inviting, highly expressive voice, slightly high pitched, with very clear audio and no background noise."
  }
 ]
}

In [ ]:
import torch, soundfile as sf, numpy as np, jiwer, re
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer, pipeline
assert not SMOKE, "needs the gated Indic Parler model: start this run from the Kaggle UI with the HF_TOKEN secret"
dev = "cuda"
model = ParlerTTSForConditionalGeneration.from_pretrained(MODEL).to(dev)
tok = AutoTokenizer.from_pretrained(MODEL)
desc_tok = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)
asr = pipeline("automatic-speech-recognition", "openai/whisper-small", device=0)
SR = model.config.sampling_rate
OUT = WORK + "/narration"; os.makedirs(OUT + "/takes", exist_ok=True)
norm = lambda s: re.sub(r"[^a-z0-9ऀ-ॿ ]", " ", s.lower()).split()
scores = {}
import librosa
def pitch_var(a):  # spread of log pitch over voiced frames: higher = livelier, less monotone
    f0, v, _ = librosa.pyin(a, fmin=70, fmax=400, sr=SR, frame_length=2048)
    f0 = f0[v & ~np.isnan(f0)]
    return float(np.std(np.log(f0))) if len(f0) > 10 else 0.0
for line in SPEC["lines"]:
    d = desc_tok(line.get("desc") or SPEC["voices"][line["voice"]], return_tensors="pt").to(dev)
    takes = []
    says = line.get("say") or [line["text"]]
    per = 3 if len(says) > 1 else 4  # 3 takes per spelling when the name is in the line
    for seed, say in [(sd, sy) for sy in says for sd in range(per)]:
        p = tok(say, return_tensors="pt").to(dev)
        torch.manual_seed(seed)
        with torch.no_grad():
            a = model.generate(input_ids=d.input_ids, attention_mask=d.attention_mask,
                               prompt_input_ids=p.input_ids, prompt_attention_mask=p.attention_mask, do_sample=True)
        a = a.cpu().float().numpy().squeeze()
        vi = says.index(say)
        f = f"{OUT}/takes/{line['id']}_v{vi}s{seed}.wav"
        sf.write(f, a, SR)
        lang = "hi" if line["voice"] in ("hinglish", "aarav") else "en"
        hyp = asr(f, generate_kwargs={"language": lang, "task": "transcribe"})["text"]
        NAME = re.compile(r"^(my|mai|mi|mee|mae|may)[a-z]*n[a-z]*$")  # any spelling of the name, ignored in WER
        strip = lambda ws: [w for w in ws if not NAME.match(w)]
        w = jiwer.wer(" ".join(strip(norm(line["text"]))) or "x", " ".join(strip(norm(hyp))) or "x") if lang == "en" else None
        name_ok = None
        if "Myna" in line["text"]:  # did it say MY-na (not Mina / Meena / Miner)?
            name_ok = bool(re.search(r"\b(my ?nah?|my ?nuh|my ?now|mine? ?(are|a|ah|uh)|mai ?nah?|mainah?|mayna|mynah?|maena)\b", hyp.lower()))
        pv = pitch_var(a)
        takes.append({"seed": seed, "variant": vi, "file": f, "name_ok": name_ok, "wer": w, "pitch_var": round(pv, 3), "dur": round(len(a) / SR, 2), "heard": hyp})
        print(line["id"], vi, seed, name_ok, w, round(pv, 3), round(len(a) / SR, 2), hyp[:90], flush=True)
    wer = lambda t: t["wer"] if t["wer"] is not None else 0
    pool = [t for t in takes if t["name_ok"] is not False] or takes  # the name must sound right, if any take manages it
    ok = [t for t in pool if wer(t) <= min(map(wer, pool)) + 0.1 and t["dur"] < 2.5 * len(line["text"]) / 14 + 1]
    best = max(ok or takes, key=lambda t: t["pitch_var"])  # most expressive among the intelligible takes
    import shutil; shutil.copy(best["file"], f"{OUT}/{line['id']}.wav")
    scores[line["id"]] = {"best": best, "takes": takes}
json.dump({"sr": SR, "scores": scores}, open(OUT + "/scores.json", "w"), indent=1, ensure_ascii=False)